In [ ]:
import sys
from pathlib import Path

parent_dir = str(Path(__file__).resolve().parent.parent) if '__file__' in globals() else str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path

# Base directory
base_dir = Path('/Data/phi2FM_n_shot')

records = []
pretrained_paths = {}

# Walk through modes, models, tasks
for mode in ['lp']:
    mode_dir = base_dir / mode
    if not mode_dir.exists():
        continue
    for model_dir in mode_dir.iterdir():
        if not model_dir.is_dir():
            continue
        
        # Initialize model entry in pretrained_paths if not exists
        model_name = model_dir.name
        if model_name not in pretrained_paths:
            pretrained_paths[model_name] = {}
        
        for task_dir in model_dir.iterdir():
            if not task_dir.is_dir():
                continue
            
            task_name = task_dir.name
            
            # nested task repetition
            nested = task_dir / task_dir.name
            if not nested.exists():
                continue
            
            best_weights_5000 = None
            latest_date_5000 = 0
            
            for run_dir in nested.iterdir():
                if not run_dir.is_dir():
                    continue
                artifacts_path = run_dir / 'artifacts.json'
                if not artifacts_path.exists():
                    continue
                
                # load json first to get n_shots
                try:
                    with open(artifacts_path, 'r') as f:
                        data = json.load(f)
                except (json.JSONDecodeError, IOError):
                    continue
                
                # Try to get n_shots from artifacts.json first
                n_shots = data.get("training_parameters", {}).get('n_shot')
                
                # If not found in artifacts, parse from folder name as fallback
                if n_shots is None:
                    shots = run_dir.name.split('_')[-1]
                    try:
                        n_shots = int(shots)
                    except ValueError:
                        n_shots = None
                
                # Only process 5000 n_shots for pretrained_paths
                if n_shots == 5000:
                    # Extract date from folder name (format: YYYYMMDD_...)
                    date_str = run_dir.name.split('_')[0]
                    try:
                        date_value = int(date_str) if date_str.isdigit() and len(date_str) == 8 else 0
                    except (ValueError, IndexError):
                        date_value = 0
                    
                    # Look for best weight files in this directory
                    best_files = list(run_dir.glob("*best*"))
                    weight_files = [f for f in best_files if f.suffix in ['.pth', '.pt', '.ckpt']]
                    
                    # If we found weight files and this is the latest date, update the path
                    if weight_files and date_value >= latest_date_5000:
                        latest_date_5000 = date_value
                        # Take the first weight file found (you can modify this logic if needed)
                        best_weights_5000 = str(weight_files[0])
                
                # Continue with the original record creation for all n_shots
                # Extract date from folder name (format: YYYYMMDD_...)
                date_str = run_dir.name.split('_')[0]
                try:
                    date_value = int(date_str) if date_str.isdigit() and len(date_str) == 8 else 0
                except (ValueError, IndexError):
                    date_value = 0
                
                metrics = data.get('test_metrics', {})
                record = {
                    'mode': mode,
                    'model': model_dir.name,
                    'task': task_dir.name,
                    'n_shots': n_shots,
                    'date_value': date_value,
                    'precision_micro': metrics.get('precision_micro'),
                    'precision_macro': metrics.get('precision_macro'),
                    "recall_micro": metrics.get('recall_micro'),
                    "recall_macro": metrics.get("recall_macro"),
                    "f1_micro": metrics.get("f1_micro"), 
                    "f1_macro": metrics.get("f1_macro"),
                    'accuracy': metrics.get('acc')
                }
                # optionally include class-wise
                for i, p in enumerate(metrics.get('precision_per_class', [])):
                    record[f'precision_class_{i}'] = p
                records.append(record)
            
            # Store the best weights path for this model-task combination
            if best_weights_5000:
                pretrained_paths[model_name][task_name] = best_weights_5000

# Create DataFrame
df = pd.DataFrame(records)

# Keep only the latest run for each combination of mode, model, task, and n_shots
if not df.empty:
    # Sort by date_value and keep the latest for each group
    df = df.sort_values('date_value').groupby(['mode', 'model', 'task', 'n_shots']).last().reset_index()
    # Remove the date_value column as it's no longer needed
    df = df.drop('date_value', axis=1)

# Print the pretrained_paths dictionary structure
print("Pretrained paths structure:")
for model, tasks in pretrained_paths.items():
    print(f"\n{model}:")
    for task, path in tasks.items():
        print(f"  {task}: {path}")

In [ ]:
from collections import OrderedDict
import os
import sys
import yaml 
from utils.load_data import load_data
from training_script import get_args, read_yaml, override_paths_with_env, main, MODELS_224, MODELS_224_r30, CNN_PRETRAINED_LIST, VIT_CNN_PRETRAINED_LIST, get_models_pretrained, get_models
from training_script import ddp_setup, module_memory_usage
from training_script import MODELS_224_r30, MODELS_224
import os
import yaml

import torch
# torch.autograd.detect_anomaly(check_nan=True)

from torchinfo import summary
from fvcore.nn import FlopCountAnalysis

import numpy as np
import random
import inspect
from collections import OrderedDict


import torch.nn as nn
from datetime import date
import argparse

from torch.nn.parallel import DistributedDataParallel as DDP



import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from models.model_Baseline import BaselineNet
from models.model_CoreCNN_versions import CoreUnet_nano, CoreUnet_tiny, CoreUnet_base, CoreUnet_large, CoreUnet_huge, Core_nano, CoreUnetGeolocation_nano
from models.model_Mixer_versions import Mixer_nano, Mixer_tiny, Mixer_base, Mixer_large, Mixer_huge
from models.model_LinearViT_versions import LinearViT_base, LinearViT_large, LinearViT_huge
from models.model_AutoEncoderViT_versions import AutoencoderViT_base, AutoencoderViT_large, AutoencoderViT_huge
from models.model_GeoAwarePretrained import MixerGeoPretrained, get_mixer_kwargs, get_core_encoder_kwargs, CoreEncoderGeoPretrained, CoreEncoderGeoPretrained_combined, CoreEncoderGeoAutoEncoder
from models.model_GeoAwarePretrained_classifier import CoreEncoderGeoPretrained_Classifier
from models.model_AutoEncoderViTPretrained import vit_cnn, vit_cnn_gc, vit_large, get_core_decoder_kwargs
from models.model_AutoEncoderViTPretrained_wSkip import vit_cnn_wSkip, vit_cnn_gc_wSkip, vit_large_wSkip
from models.model_AutoEncoderViTPretrained_classifier import vit_cnn_classifier, vit_cnn_gc_classifier
from models.model_CoreVAE import CoreVAE_nano
from models.model_SatMAE import satmae_vit_cnn
from models.models_Prithvi import prithvi
from models.model_Seco import seasonal_contrast
from models.model_Resnet50 import resnet
from models.code_phileo_precursor.model_foundation_local_rev2 import PhileoPrecursor, PhileoPrecursorClassifier
from models.model_moco_ssl4eo12 import moco_resnet
from models.model_dino_ssl4eo12 import dino_resnet

from pretrain.models.utils_fm import get_phisat2_model
from downstream.models.phisatnet_downstream import PhiSatNetDownstream
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib
import scienceplots
from scipy.stats import gaussian_kde
from itertools import islice

from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import matplotlib.patches as patches

import numpy as np
import pandas as pd

import os
import shutil

from tqdm import tqdm

from utils.load_data import load_data
from utils.training_utils import read_yaml
from utils.utils import module_memory_usage, dataloader_to_arrays, dataloader_to_tensors, convert_to_onnx, ddp_setup, ddp_cleanup

from utils.training_utils import read_yaml
from training_script import main
from matplotlib.colors import BoundaryNorm

from glob import glob
from utils.data_protocol_utils import sanity_check_labels_exist
from utils.load_data import load_data
from training_script import get_models_pretrained

import torch
torch.set_default_device('cuda')
torch.manual_seed(123456)

# --------------------------------------------------------------
# 0.  Colour map + class names + plotters (Unchanged)
# --------------------------------------------------------------
mpl.rcParams.update({
    'font.size': 28,        # ─── base font size for *all* text ───
})

legend_fontsize = 25
out = {}
out_label = {}
# for key in out_label.keys():
#     out_label[key] = {}
#     for task in ['lc', 'lc_classification', 'building', 'roads']:
#         out_label[key][task] = None

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_list = ['caco', 'dino', 'gassl', 'geoaware', 'moco', 'phisatnet', 'prithvi', 'satmae', 'seco', 'uniphi']
task_list = ['lc', 'lc_classification', 'building', 'roads']

dataset_dir = { 
     "fire": "/Data/fire_dataset/fire_dataset.zarr", 
     "burned_area": "/Data/lpl_burned_area/burned.zarr", 
     "clouds": "/Data/phisatnet_clouds/phisatnet_clouds.zarr", 
     "worldfloods": "/Data/worldfloods/worldfloods.zarr", 
}    
labels = {
     "fire": ['safe', 'fire', 'burnt', 'water'],
     "burned_area": ['Background', 'Burned Area', 'Clouds', 'Waterbodies'],
     "clouds": ['No cloud', 'Cloud', 'value2', 'value3', 'value4'],
     "worldfloods": ['Clouds', 'Land', 'Water'],
}
class_labels = labels["fire"]
model_names = {
    "CaCO": "phi2_caco",
    "DINO": "phi2_dino", 
    "GASSL": "phi2_gassl", 
    "GeoAware": "phi2_GeoAware", 
    "MoCo": "phi2_moco", 
    "PhisatNet": "phisatnet_downstream", 
    "Prithvi 1.0": "phi2_prithvi", 
    "SatMAE": "phi2_SatMAE", 
    "SeCo": "phi2_seasonal_contrast", 
    "UniPhi": "phi2_phileo_precursor", 
}
model_configs = {
    #  "GeoAware": "geoaware",
    #  "CaCO": "caco",
    #  "DINO": "dino",
    #  "GASSL": "gassl",
    #  "MoCo": "moco",
    #  "PhisatNet": "phisatnet",
    #  "Prithvi 1.0": "prithvi",
    #  "SatMAE": "satmae",
     "SeCo": "seco",
     #"UniPhi": "uniphi",
}
configs_dirs = {
     "fire": "args/finetune_FMs/fire",
     "burned_area": "args/finetune_FMs/lpl_burned_area",
     "clouds": "args/finetune_FMs/phisatnet_clouds",
     "worldfloods": "args/finetune_FMs/worldfloods",
}
map_code_to_color = {
    0 : (0  , 100,   0),    # Tree cover
    1 : (255, 187,  34),    # Shrubland
    2 : (255, 255,  76),    # Grassland
    3 : (240, 150, 255),    # Cropland
    4 : (250,   0,   0),    # Built-up
    5 : (180, 180, 180),    # Bare / sparse vegetation
    6 : (240, 240, 240),    # Snow and Ice
    7 : (0  , 100, 200),    # Permanent water bodies
    8 : (0  , 150, 160),    # Herbaceous wetland
    9 : (0  , 207, 117),    # Mangroves
    10: (250, 230, 160),    # Moss and lichen
}



codes     = sorted(map_code_to_color)
lc_colors = [tuple(c/255 for c in map_code_to_color[i]) for i in codes]
lc_cmap   = ListedColormap(lc_colors, name="landcover")

colors = {
    'fire':  [tuple(c/255 for c in map_code_to_color[i]) for i in codes],
    'worldfloods':  [tuple(c/255 for c in map_code_to_color[i]) for i in codes],
    'clouds': [tuple(c/255 for c in map_code_to_color[i]) for i in codes],
    'burned_area': [tuple(c/255 for c in map_code_to_color[i]) for i in codes],
}
color_maps = {
    'fire':  ListedColormap(colors['fire'], name="landcover"),
    'worldfloods':  ListedColormap(colors['worldfloods'], name="landcover"),
    'clouds': ListedColormap(colors['clouds'], name="landcover"),
    'burned_area': ListedColormap(colors['burned_area'], name="landcover"),
}
# bounds = np.arange(len(lc_colors)+1) - 0.5
bounds = {
    'fire': np.arange(len(colors['fire'])+1) - 0.5,
    'worldfloods': np.arange(len(colors['worldfloods'])+1) - 0.5,
    'clouds': np.arange(len(colors['clouds'])+1) - 0.5,
    'burned_area': np.arange(len(colors['burned_area'])+1) - 0.5,
}
# norm   = BoundaryNorm(bounds, ncolors=len(lc_colors))
norm = {
    'fire': BoundaryNorm(bounds['fire'], ncolors=len(colors['fire'])),
    'worldfloods': BoundaryNorm(bounds['worldfloods'], ncolors=len(colors['worldfloods'])),
    'clouds': BoundaryNorm(bounds['clouds'], ncolors=len(colors['clouds'])),
    'burned_area': BoundaryNorm(bounds['burned_area'], ncolors=len(colors['burned_area'])),
}


# --------------------------------------------------------------
# 1.  Individual plotting helpers (Unchanged)
# --------------------------------------------------------------

def show_segmentation(ax, tensor):
    """Segmentation map (ground-truth [1,H,W] or argmax predictions [C,H,W])."""
    #print(f"Before showing: Tensor shape: {tensor.shape}, dtype: {tensor.dtype}, min: {tensor.min()}, max: {tensor.max()}")
    if tensor.ndim == 3 and tensor.shape[2] < tensor.shape[1] and tensor.shape[2] < tensor.shape[0]:
        tensor = tensor.permute(2, 0, 1)  # Change to (C,H,W)
    if tensor.ndim == 3 and tensor.shape[0] > 1: # Check if it's prediction tensor (C,H,W)
        tensor = tensor.argmax(0) # Get class indices
    #print(f"After argmax (if applied): Tensor shape: {tensor.shape}, dtype: {tensor.dtype}, min: {tensor.min()}, max: {tensor.max()}")
    tensor = tensor.squeeze().cpu() # Squeeze batch/channel dims -> (H,W)
    #print(f"After squeeze and cpu: Tensor shape: {tensor.shape}, dtype: {tensor.dtype}, min: {tensor.min()}, max: {tensor.max()}")
    #print(f"Tensor after show preparation: {tensor}")
    ax.imshow(tensor, cmap=lc_cmap, norm=norm['fire'], interpolation='nearest')
    ax.axis("off")

def show_single_label(ax, tensor, thresh=0.5):
    """
    Display an 11-element multilabel / probability vector as a 3×4 grid.
    Input tensor shape: [11] or [1, 11]
    """
    if tensor.ndim > 1:
        tensor = tensor.squeeze() # Ensure tensor is 1D

    # Ensure tensor is on CPU before numpy conversion
    gt_class_idx = tensor.cpu()
    #print(f"Vals = {vals}")
    #print(f"Chosen = {chosen}")
    rows, cols = 2, 2 #4, 3
    cell_w, cell_h = 1/cols, 1/rows

    for idx in range(rows * cols):
        r, c = divmod(idx, cols)
        x0, y0 = c*cell_w, 1 - (r+1)*cell_h

        face = lc_colors[idx] if idx < len(class_labels) else 'white'
        rect = patches.Rectangle(
            (x0, y0), cell_w, cell_h,
            facecolor=face, edgecolor="black", linewidth=0.5,
            transform=ax.transAxes
        )
        ax.add_patch(rect)

        # if idx == rows*cols - 1:
        #     ax.text(
        #         x0 + cell_w/2, y0 + cell_h/2, "n/a",
        #         ha="center", va="center",
        #         fontsize=legend_fontsize,
        #         color="gray", fontstyle="italic",
        #         transform=ax.transAxes
        #     )
        #     continue

        # Check index bounds before accessing 'chosen'
        if gt_class_idx == idx: #idx < len(chosen) and chosen[idx]:
            pad = 0.1 * min(cell_w, cell_h)
            line1 = plt.Line2D(
                [x0+pad, x0+cell_w-pad], [y0+pad, y0+cell_h-pad],
                transform=ax.transAxes, linewidth=2, color="k"
            )
            line2 = plt.Line2D(
                [x0+pad, x0+cell_w-pad], [y0+cell_h-pad, y0+pad],
                transform=ax.transAxes, linewidth=2, color="k"
            )
            ax.add_line(line1)
            ax.add_line(line2)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')

def show_multilabel(ax, tensor, thresh=0.5):
    """
    Display an 11-element multilabel / probability vector as a 3×4 grid.
    Input tensor shape: [11] or [1, 11]
    """
    if tensor.ndim > 1:
        tensor = tensor.squeeze() # Ensure tensor is 1D

    # Ensure tensor is on CPU before numpy conversion
    tensor = tensor.cpu()

    needs_sigmoid = (tensor.min() < 0) or (tensor.max() > 1)
    # Apply sigmoid only if values are outside [0, 1] range (likely logits)
    vals = (torch.sigmoid(tensor.float()).numpy()
            if needs_sigmoid else tensor.float().numpy())
    chosen = vals > thresh
    #print(f"Vals = {vals}")
    #print(f"Chosen = {chosen}")
    rows, cols = 2, 2 #4, 3
    cell_w, cell_h = 1/cols, 1/rows

    for idx in range(rows * cols):
        r, c = divmod(idx, cols)
        x0, y0 = c*cell_w, 1 - (r+1)*cell_h

        face = lc_colors[idx] if idx < len(class_labels) else 'white'
        rect = patches.Rectangle(
            (x0, y0), cell_w, cell_h,
            facecolor=face, edgecolor="black", linewidth=0.5,
            transform=ax.transAxes
        )
        ax.add_patch(rect)

        # if idx == rows*cols - 1:
        #     ax.text(
        #         x0 + cell_w/2, y0 + cell_h/2, "n/a",
        #         ha="center", va="center",
        #         fontsize=legend_fontsize,
        #         color="gray", fontstyle="italic",
        #         transform=ax.transAxes
        #     )
        #     continue

        # Check index bounds before accessing 'chosen'
        if chosen == idx: #idx < len(chosen) and chosen[idx]:
            pad = 0.1 * min(cell_w, cell_h)
            line1 = plt.Line2D(
                [x0+pad, x0+cell_w-pad], [y0+pad, y0+cell_h-pad],
                transform=ax.transAxes, linewidth=2, color="k"
            )
            line2 = plt.Line2D(
                [x0+pad, x0+cell_w-pad], [y0+cell_h-pad, y0+pad],
                transform=ax.transAxes, linewidth=2, color="k"
            )
            ax.add_line(line1)
            ax.add_line(line2)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal', adjustable='box')
    ax.axis('off')

def show_building(ax, tensor):
    """Building-density regression (YlOrRd). Input shape [1,H,W] or [1,1,H,W]"""
    ax.imshow(tensor.squeeze().cpu(), cmap='YlOrRd')
    ax.axis("off")

def show_roads(ax, tensor):
    """Road-density regression (PuBu). Input shape [1,H,W] or [1,1,H,W]"""
    ax.imshow(tensor.squeeze().cpu(), cmap='PuBu')
    ax.axis("off")
def sigmoid_np(z):
    return 1.0 / (1.0 + np.exp(-z))

def render_false_rgb(arr, channel_first):
    if channel_first:
        if arr.ndim == 3 and arr.shape[2] < arr.shape[1] and arr.shape[2] < arr.shape[0]:
            arr = np.transpose(arr, (2, 0, 1))
    else: 
        if arr.ndim == 3 and arr.shape[0] < arr.shape[1] and arr.shape[0] < arr.shape[2]:
            arr = np.transpose(arr, (1, 2, 0))
    if arr.ndim != 3 or arr.shape[2] < 3:
        raise ValueError(f"Expected ≥3 bands for RGB, got shape {arr.shape}")
    # use bands [2,1,0] as pseudo-RGB
    rgb = arr[:, :, [2, 1, 0]].astype(np.float32)
    mn, mx = rgb.min(), rgb.max()
    if mx > mn:
        rgb = (rgb - mn) / (mx - mn) 
    else:
        rgb = np.zeros_like(rgb)
    return rgb
def show_image(ax, tensor):
    """RGB composite from a multi-spectral stack. Input shape [C,H,W] or [1,C,H,W]"""
    rgb = render_false_rgb(tensor, channel_first=False)
    ax.imshow(rgb)
    ax.axis("off")
    ax.grid(False)
    
    # if tensor.ndim == 4:                      # (1,C,H,W) → (C,H,W)
    #     tensor = tensor.squeeze(0)

    # # Ensure tensor is on CPU before processing
    
    # if not isinstance(tensor, np.ndarray):
    #     tensor = tensor.cpu()
    # else: 
    #     tensor = torch.from_numpy(tensor)
    # # Assuming RGB are channels 2, 1, 0 based on original code
    # # Verify this matches your 'img' tensor's channel order (e.g., Sentinel-2 B4,B3,B2)
    # # If Sentinel-2 (B2,B3,B4,...), use indices [3, 2, 1] for RGB (B4=Red, B3=Green, B2=Blue)
    # rgb_indices = [2, 1, 0] # <<<### ADJUST THIS if your channel order is different ###>>>

    # if tensor.shape[0] >= max(rgb_indices) + 1: # Check if required channels exist
    #     rgb = tensor[rgb_indices].permute(1, 2, 0).float()   # (H,W,3)
    #     # Robust quantile calculation for normalization/clamping
    #     q_low = max(0.0, min(1.0, 0.02))
    #     q_high = max(0.0, min(1.0, 0.98))
    #     if rgb.numel() > 0:
    #         # Calculate quantiles per channel if needed, or across all pixels
    #         # Using overall quantiles for simplicity here:
    #         lo = torch.quantile(rgb, q_low)
    #         hi = torch.quantile(rgb, q_high)
    #         # Avoid division by zero if hi == lo
    #         rgb = torch.clamp((rgb - lo) / (hi - lo + 1e-6), 0, 1)
    #     else:
    #         rgb = torch.zeros_like(rgb) # Handle empty tensor case
    #     ax.imshow(rgb)
    # elif tensor.shape[0] == 1: # Grayscale fallback if only 1 channel
    #      img_gray = tensor.squeeze(0).float()
    #      lo, hi = torch.quantile(img_gray, 0.02), torch.quantile(img_gray, 0.98)
    #      img_gray = torch.clamp((img_gray - lo) / (hi - lo + 1e-6), 0, 1)
    #      ax.imshow(img_gray, cmap='gray')
    # else: # Fallback for insufficient channels
    #     ax.text(0.5, 0.5, f"Cannot display RGB\n{tensor.shape[0]} channels",
    #             ha='center', va='center', transform=ax.transAxes)

    # ax.axis("off")


# Map each downstream task to its plotting helper (Unchanged)
plotters = {
    "img"              : show_image,
    "lc"               : show_segmentation,
    "lc_classification": show_multilabel,
    "building"         : show_building,
    "roads"            : show_roads,
    "fire"             : show_single_label, 
    "burned_area"      : show_segmentation,
    "clouds"           : show_segmentation, 
    "worldfloods"      : show_segmentation,  
}

# Task Labels (Unchanged)
task_labels = {
    "img"               : "Input Image",
    "lc"                : "Land Cover\nSegmentation\n(Pixel-Level)",
    "lc_classification" : "Land Cover\nClassification\n(Image-Level)",
    "building"          : "Building Density\nRegression\n(Pixel-Level)",
    "roads"             : "Road Density\nRegression\n(Pixel-Level)",
    "fire"             : "Fire\nClassification\n(Image-Level)", 
    "burned_area"      : "Burned Area\nSegmentation\n(Pixel-Level)",
    "clouds"           : "Clouds\nSegmentation\n(Pixel-Level)", 
    "worldfloods"      : "World Floods\nSegmentation\n(Pixel-Level)", 
}

# Model Labels (Unchanged - assuming these are still relevant)
model_labels = {
    "caco"      : "CaCO",
    "phisatnet" : "PhiSatNet",
    "geoaware"  : "GeoAware",
    "dino"      : "DINO",
    "gassl"     : "GASSL",
    "moco"      : "MoCo",
    "prithvi"   : "Prithvi 1.0",
    "satmae"    : "SatMAE",
    "seco"      : "SeCo",
    "uniphi"    : "UniPhi",
}

# Row order (down-stream tasks) (Unchanged)
task_order = ["img", "fire", "burned_area", "worldfloods", "clouds"]

In [ ]:


def get_model_and_dataset(experiment_name, downstream_task, model_name, augmentations, batch_size, model_device, generator_device, num_workers, early_stop, 
        epochs, input_channels, output_channels, input_size, lr, lr_scheduler, n_shot, split_ratio, regions, vis_val, warmup, warmup_steps,
        warmup_gamma, pretrained_model_path, freeze_pretrained, data_path_128_10m, data_path_224_10m, data_path_224_30m, data_path_inference_128, 
        data_path_inference_224, train_mode, downstream_model_path, output_path, data_parallel, 
        device_ids, only_get_datasets, pad_bands, min_lr):
    """ 
    main script for PhilEO Bench. Used to run model training experiments with randomly initialized and pre-trained models on a number of downstream tasks. 
    The script handles dataset creation (based on data protocol options selected), data preprocessing (based on downstream task & model type) & model, training, validation and testing. 

    Parameters
    ----------
        experiment_name (str): Experiment name
        downstream_task (str): Select downstream task to test, validate and test on. Options: {DOWNSTREAM_LIST}
        model_name (str): Select model. Options:{MODEL_LIST}
        augmentations (bool, optional): Toggle on/off basic data augmentations (Rotation, Mirror, Noise). Defaults to False.
        batch_size (int, optional): Define training batch size. Defaults to 16.
        model_device (_type_, optional): Select model device. Defaults to torch.device('cuda' if torch.cuda.is_available() else 'cpu').
        generator_device (_type_, optional): Select dataloader device. Defaults to torch.device('cuda' if torch.cuda.is_available() else 'cpu').
        num_workers (int, optional): Select number of workers for dataloader. Defaults to 4.
        early_stop (int, optional):Define early stoping patience. Defaults to 25.
        epochs (int, optional): Define number of training epochs. Defaults to 250.
        input_channels (int, optional): Define number of data input channels. Defaults to 10.
        output_channels (int, optional): Define number of model output channels. Defaults to 1.
        input_size (int, optional): Define data input size. Defaults to 128.
        lr (float, optional): Define optimizer learning rate. Defaults to 0.001.
        lr_scheduler (str, optional): Define learning rate scheduler. Options: [None, 'reduce_on_plateau', 'cosine_annealing']. Defaults to None.
        n_shot (int, optional): Define dataset protocol - n samples per region. Defaults to None.
        split_ratio (float, optional): Define dataset protocol - percentage of full dataset. Defaults to 0.1.
        regions (list, optional): Select regions to include in training and test sets. If no regions are defined (None) all avalible regions will be included
                                  Options: [None, 'denmark-1', 'denmark-2', 'east-africa', 'egypt-1', 'eq-guinea', 'europe', 'ghana-1',
                                 'isreal-1', 'isreal-2', 'japan', 'nigeria', 'north-america', 'senegal', 'south-america',
                                 'tanzania-1', 'tanzania-2', 'tanzania-3', 'tanzania-4', 'tanzania-5', 'uganda-1'] Defaults to None.
        vis_val (bool, optional): If set to True data visulisations will be generated at each validation step. Defaults to True.
        warmup (bool, optional): If set to True a linear optimizer warmup phase will occour. Defaults to False.
        warmup_steps (int, optional): Define number of steps for linear warmup phase. Defaults to 5.
        warmup_gamma (int, optional): Define learning rate increase per step in linear warmup phase - new_lr = lr*gamma. Defaults to 10. N.B. initial lr is calulated as follows init_lr = lr/(gamma**warmup_steps)
        pretrained_model_path (str, optional): For pretrained models define the model weights path. Defaults to None.
        freeze_pretrained (bool, optional): If True pretrained encoder weights will be frozen during training. Defaults to None.
        data_path_128_10m (str, optional): Define data path for 128x128 10m resolution dataset. Defaults to None.
        data_path_224_10m (str, optional): Define data path for 224x224 10m resolution dataset. Defaults to None.
        data_path_224_30m (str, optional): Define data path for 224x224 30m resolution dataset. Defaults to None.
        data_path_inference_128 (str, optional): Define data path for inference data of size 128. Defaults to None.
        data_path_inference_224 (str, optional): Define data path for inference data of size 224. Defaults to None.
        train_mode (str, optional): Define if only inference should be run. Options: ['yes', 'no', 'only']. Defaults to None.
        downstream_model_path (str, optional): Define model path for inference. Defaults to None.
        output_path (str, optional): Define folder to save artifacts in. Defaults to None.
        data_parallel (str, optional): If set to True Model training will be parallized on multiple gpus. Defaults to None.
        device_ids (list, optional): Define GPU IDs to use for parallization. Defaults to None.
        only_get_datasets (bool, optional): If set to True only datasets will be created, but no training will occur. Defaults to False.
        pad_bands (int, optional): To what number of bands to pad. Defaults to 10.
        min_lr (float, optional): Define minimum learning rate for cosine annealing scheduler and warmup. Defaults to 1e-6.
    """

    # -----------------------------------------------------------------------
    # 1. Multi GPU Setup (DDP, DP, or False) -- DDP NOT FULLY IMPLEMENTED FOR DOWNSTREAM
    # -----------------------------------------------------------------------
    print(f"Using device {model_device}")
    if data_parallel == 'DDP':
        world_rank, local_rank, world_size = ddp_setup()
        device = torch.device(f'cuda:{device_ids[local_rank]}' if torch.cuda.is_available() else 'cpu')
        torch.cuda.set_device(device)
        model_device, generator_device = device, 'cpu'
        print(f'Using DDP: rank {world_rank}/{world_size}, device {device}')
    else:
        world_rank, local_rank, world_size = 0, 0, 1
        if model_device == 'cuda':
            model_device = f'cuda:{device_ids[0]}' if device_ids else 'cuda'
        torch.set_default_device(model_device)
        generator_device = model_device
        print(f'Device (not using DDP): {model_device}')

    if torch.cuda.device_count() > 1 and world_rank == 0:
        num_gpus = torch.cuda.device_count() if device_ids is None else len(device_ids)
        print(f"Let's use {num_gpus} GPUs!")


    # -----------------------------------------------------------------------
    # 2. DEFINE THE MODEL
    # -----------------------------------------------------------------------
    
    # LOAD PRETRAINED MODEL
    if pretrained_model_path is not None:
        if world_rank == 0:
            print('model_name: ', model_name)
        assert model_name in (CNN_PRETRAINED_LIST + VIT_CNN_PRETRAINED_LIST), f"Pretrained weights were given but model {model_name} not found in list of pretrained models: {CNN_PRETRAINED_LIST + VIT_CNN_PRETRAINED_LIST}"
        assert freeze_pretrained is not None, f"When supplying a pretrained model 'freeze_pretrained' must be either True or False"
        model = get_models_pretrained(model_name, input_channels, output_channels, input_size, path_model_weights=pretrained_model_path, freeze=freeze_pretrained)
        if model_name == 'GeoAware_contrastive_core_nano' or model_name == 'GeoAware_contrastive_core_nano_classifier':
            NAME = model.__class__.__name__ +'_contrastive_frozen' if freeze_pretrained else model.__class__.__name__ +'_contrastive_unfrozen'
        elif model_name == 'GeoAware_mh_pred_core_nano' or model_name == 'GeoAware_mh_pred_core_nano_classifier':
            NAME = model.__class__.__name__ +'_mh_pred_frozen' if freeze_pretrained else model.__class__.__name__ +'_mh_pred_unfrozen'
        else:
            NAME = model.__class__.__name__ + '_frozen' if freeze_pretrained else model.__class__.__name__ + '_unfrozen'
        if world_rank == 0:
            print(f'Loaded pretrained model: {model_name} with {NAME} weights')

    # LOAD RANDOMLY INITIALIZED MODEL
    else:
        if freeze_pretrained:
            if world_rank == 0:
                print(f"Ignoring freeze_pretrained set to {freeze_pretrained} as no pretrained model was supplied")
        model = get_models(model_name, input_channels, output_channels, input_size)
        NAME = model.__class__.__name__

    # If want to load weights of full downstream model, not just a feature extractor
    if downstream_model_path:
        print('\n\n------------------------------------------------------------------------')
        print(f'WARNING: IGNORING pretrained_model_path. Inference model path given. Full downstream model will be loaded.')
        print('------------------------------------------------------------------------\n\n')
        
        assert model is not None, "This model implementation requires pretrained weights to be loaded first, even if they will be overwritten"
        
        state_dict = torch.load(downstream_model_path)
        if world_rank == 0:
            print(f'Loading inference model from {downstream_model_path}')

        new_state_dict = OrderedDict()
        
        for key, value in state_dict.items():
            # Remove 'module.' prefix if it exists
            new_key = key.replace("module.", "")
            new_state_dict[new_key] = value

        # Load the modified state dictionary into the model
        model.load_state_dict(new_state_dict, strict=True)
    
    model = model.to(model_device)

    # Parallelize model (DP or DDP) and print model summary
    if data_parallel == 'DP':
        model = nn.DataParallel(model, device_ids=device_ids).to(model_device)
    elif data_parallel == 'DDP':
        model = nn.SyncBatchNorm.convert_sync_batchnorm(model).to(model_device)
        model = DDP(model, device_ids=[model_device], output_device=model_device)

    # -----------------------------------------------------------------------
    # 4. Load datasets
    # -----------------------------------------------------------------------

    # Validate configuration
    task_output_channels = {
        'lc': 11,
        'lc_classification': 11,
        'roads': 1,
        'building': 1,
        'building_classification': 5,
        'roads_classification': 2,
        'coords': 3, 
        'fire': 4, 
        'burned_area':4, 
        'clouds': 5, 
        'worldfloods': 3
    }
    assert output_channels == task_output_channels[downstream_task], (
        f"{downstream_task} tasks should have {task_output_channels[downstream_task]} output channels, it has {output_channels}."
    )
    assert n_shot is not None or split_ratio is not None, "Please define data partition protocol!"
    assert isinstance(n_shot, int) ^ isinstance(split_ratio, float) , "n_shot cannot be used with split_ratio!"


    # Choose dataset path based on model
    if model_name in MODELS_224_r30:
        dataset_name, dataset_folder, data_path_inference = '224_30m', data_path_224_30m, data_path_inference_224
    elif model_name in MODELS_224:
        dataset_name, dataset_folder, data_path_inference = '224_10m', data_path_224_10m, data_path_inference_224
    else:
        dataset_name, dataset_folder, data_path_inference = '128_10m', data_path_128_10m, data_path_inference_128
        
    # Determine if cropping (model requires smaller images), if use by region (prob only relevant to PhilEO-Bench), 
    # and set weights or pos_weight for loss function
    crop_images = True if model_name == 'phileo_precursor' or model_name == 'phileo_precursor_classifier' else False
    by_region = False if downstream_task == 'coords' else True
    pos_weight, weights = None, None

    
    # Data partition
    if isinstance(n_shot, int):
        if n_shot == 0:
            n_shot = 1
            train_mode = 'inference'

        additional_params = {'n':n_shot, 'regions':regions, 'y':downstream_task, 'data_selection':'create', 'name':dataset_name,'crop_images':crop_images}

    elif isinstance(split_ratio, float):

        additional_params = {'split_percentage':split_ratio, 'regions':regions, 'y':downstream_task,'by_region':by_region}

    # Create dataloaders
    print(f'Batch size: {batch_size}')
    if not downstream_task == "fire":
        patch_size = (256,256)
    else:
        patch_size = None
    weights, pos_weight, _, dl_test, _, _= load_data(
        dataset_folder,
        with_augmentations=augmentations,
        num_workers=num_workers,
        batch_size=batch_size,
        downstream_task=downstream_task,
        model_name=model_name.split('_')[0],
        device=generator_device,
        pad_bands=pad_bands,
        crop_images=crop_images, 
        num_classes=output_channels, 
        n=n_shot, 
        weights_dir=downstream_task, 
        patch_size=patch_size
    )
    
    return model, dl_test

def return_inference_visualization(images, labels, task: str, model):
    model.eval()
    
    # Get the device where the model is lsocated
    model_device = next(model.parameters()).device
    if not isinstance(images, torch.Tensor):
        images = torch.tensor(images)
    # Ensure inputs are on the same device as the model
    images = images.to(model_device)
    
    if isinstance(labels, torch.Tensor):
        labels = labels.to(model_device)
    else:
        labels = torch.tensor(labels).to(model_device)
    if images.dim() == 3:
        images = images.unsqueeze(0) 
    with torch.no_grad():
        outputs = model(images)

        if task == 'burned_area' or task == 'clouds' or task == 'worldfloods' or task == 'fire':
            if isinstance(outputs, torch.Tensor):
                outputs_labels = torch.argmax(outputs, dim=1)  # [B, H, W]
            else:
                outputs_tensor = torch.from_numpy(outputs).to(model_device)
                outputs_labels = torch.argmax(outputs_tensor, dim=1)
            
            if isinstance(labels, torch.Tensor) and labels.dim() > 2:
                labels = torch.argmax(labels, dim=1)  # [B, H, W]
            
            # Move back to CPU for visualization
            outputs_labels = outputs_labels.cpu()
            labels = labels.cpu()
            
        
        # Move images to CPU for preprocessing
        images_cpu = images.cpu()
        images_processed = preprocess_image_for_rgb_display(images_cpu, channel_first=True, rgb_indices=[2, 1, 0])

        return images_processed, outputs_labels.cpu() if isinstance(outputs_labels, torch.Tensor) else outputs_labels

def preprocess_image_for_rgb_display(tensor, channel_first=True, rgb_indices=None, quantile_range=(0.02, 0.98)):
    """
    Preprocess a multi-spectral tensor for RGB display in matplotlib.
    
    Args:
        tensor: Input tensor of shape [C,H,W], [1,C,H,W], or [B,C,H,W]
        channel_first: Whether channels are first (True) or last (False)
        rgb_indices: List of 3 channel indices to use for RGB. Default: [2,1,0]
        quantile_range: Tuple of (low, high) quantiles for normalization. Default: (0.02, 0.98)
    
    Returns:
        numpy.ndarray: RGB image array of shape [H,W,3] ready for matplotlib, 
                      normalized to [0,1] range, or None if preprocessing fails
    """
    import torch
    import numpy as np
    
    # Convert to tensor if needed and ensure it's on CPU
    if isinstance(tensor, np.ndarray):
        tensor = torch.from_numpy(tensor)
    elif isinstance(tensor, torch.Tensor):
        tensor = tensor.detach().cpu()
    else:
        raise ValueError(f"Input must be numpy array or torch tensor, got {type(tensor)}")
    
    # Handle batch dimension
    if tensor.ndim == 4:  # [B,C,H,W] → [C,H,W] (take first sample)
        tensor = tensor[0]
    elif tensor.ndim == 2:  # [H,W] → return grayscale handling
        return _handle_grayscale(tensor, quantile_range)
    elif tensor.ndim != 3:
        raise ValueError(f"Expected 2D, 3D, or 4D tensor, got {tensor.ndim}D with shape {tensor.shape}")
    
    # Handle channel order
    if not channel_first and tensor.ndim == 3:
        tensor = tensor.permute(2, 0, 1)  # [H,W,C] → [C,H,W]
    
    # Default RGB indices (adjust based on your data)
    if rgb_indices is None:
        rgb_indices = [2, 1, 0]  # Assuming bands are ordered for Sentinel-2-like data
    
    # Check if we have enough channels for RGB
    if tensor.shape[0] < max(rgb_indices) + 1:
        if tensor.shape[0] == 1:
            return _handle_grayscale(tensor.squeeze(0), quantile_range)
        else:
            print(f"Warning: Only {tensor.shape[0]} channels available, need at least {max(rgb_indices)+1} for RGB")
            return None
    
    # Extract RGB channels and convert to [H,W,3]
    try:
        rgb = tensor[rgb_indices].permute(1, 2, 0).float()  # [H,W,3]
    except IndexError as e:
        print(f"Error extracting RGB channels {rgb_indices} from tensor with {tensor.shape[0]} channels: {e}")
        return None
    
    # Normalize using quantiles
    if rgb.numel() > 0:
        q_low = max(0.0, min(1.0, quantile_range[0]))
        q_high = max(0.0, min(1.0, quantile_range[1]))
        
        lo = torch.quantile(rgb, q_low)
        hi = torch.quantile(rgb, q_high)
        
        # Avoid division by zero
        if hi > lo:
            rgb = torch.clamp((rgb - lo) / (hi - lo), 0, 1)
        else:
            rgb = torch.zeros_like(rgb)
    else:
        rgb = torch.zeros_like(rgb)
    
    return rgb.numpy()

def _handle_grayscale(tensor_2d, quantile_range):
    """Helper function to handle grayscale images"""
    import torch
    
    img_gray = tensor_2d.float()
    q_low = max(0.0, min(1.0, quantile_range[0]))
    q_high = max(0.0, min(1.0, quantile_range[1]))
    
    if img_gray.numel() > 0:
        lo = torch.quantile(img_gray, q_low)
        hi = torch.quantile(img_gray, q_high)
        img_gray = torch.clamp((img_gray - lo) / (hi - lo + 1e-6), 0, 1)
    else:
        img_gray = torch.zeros_like(img_gray)
    
    # Convert grayscale to RGB by repeating channels
    rgb = torch.stack([img_gray, img_gray, img_gray], dim=-1)  # [H,W,3]
    return rgb.numpy()

    
def get_inference_model_and_dataset(model_name: str, task_name:str, yaml_config_file: str):

    # 1. Reading YAML file
    args = read_yaml(yaml_config_file)

    # 2. Run main function
    n_shot = 5000
    base_exp_name =  args.experiment_name

    args.n_shot = n_shot
    for freeze_pretrained in [True]:
    #for freeze_pretrained in [True]:
        args.freeze_pretrained = freeze_pretrained
        if freeze_pretrained:
            prefix="finetuning/"
        else:
            prefix="lp/"
        args.experiment_name = prefix+ base_exp_name
        if model_name not in model_names:
            print(f"Warning: No model named '{model_name}' found in {model_names.keys()}")
        if model_names[model_name] not in pretrained_paths:
            print(f"Warning: No pretrained model named '{model_names[model_name]}' found for {model_names[model_name]}")
        print(f"Available tasks for {model_name}: {list(pretrained_paths[model_names[model_name]].keys())}")
        args.downstream_model_path = pretrained_paths[model_names[model_name]][task_name]
        print(f"Running inference on {args.downstream_model_path}: ")
        print(f"mode: {'lp' if args.freeze_pretrained else 'finetuning'}, downstream_task: {args.downstream_task}, model_name: {args.model_name}")
        model, dl_test = get_model_and_dataset(**vars(args))
        return model, dl_test
            


In [ ]:
import pickle
import copy 

def make_inference(cache_path="inference_cache.pkl"):
    # Check if cached results exist
    
    if os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            cache = pickle.load(f)
        print(f"Loaded inference results from {cache_path}")
        #return cache
    else:
        cache = {}
    cache_entry =             {
        "imgs": {}, 
        "out_preds": {
            "fire": {}, 
            "worldfloods": {},
            "burned_area": {},
            "clouds": {}
        }, 
        "out_labels":  {
            "fire": {}, 
            "worldfloods": {},
            "burned_area": {},
            "clouds": {}
        }
    }


    sample_nums = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000]

    for task, conf_dir in configs_dirs.items():
        first = True
        for model_final_name, model_name in model_configs.items():
            model, dl_test = None, None
            for sample_num in sample_nums: 
                if sample_num not in cache.keys():
                    cache[sample_num] = copy.deepcopy(cache_entry)
                if model_name not in cache[sample_num]['out_preds'][task].keys() or cache[sample_num]['imgs'][task] is None:
                    if model is None and dl_test is None:
                        configs_file = os.path.join(os.getcwd(), conf_dir, f"{model_name}.yml")
                        model, dl_test = get_inference_model_and_dataset(model_final_name, task, str(configs_file))
                    # try:
                    img = dl_test.dataset[sample_num]['img']
                    out_label = dl_test.dataset[sample_num]['label']
                    image, out_pred = return_inference_visualization(img, out_label, task, model)
                    #if first:
                    # print(f"Shape of image for task {task}: {image.shape}")
                    # cache[sample_num]['imgs'][task] = image
                    # cache[sample_num]['out_labels'][task] = out_label
                    #first = False
                    cache[sample_num]['out_preds'][task][model_name] = out_pred
                    # except Exception as e:
                    #     print(f"Error loading model {model_name} from {configs_file}: {e}")  
                              
    # imgs = {sample_id: cache_e['imgs'] for sample_id, cache_e in cache.items()}
    # out_preds = {sample_id: cache_e['out_preds'] for sample_id, cache_e in cache.items()}
    # out_labels = {sample_id: cache_e['out_labels'] for sample_id, cache_e in cache.items()}
    #cache = {"imgs": imgs, "out_preds": out_preds, "out_labels": out_labels}
    with open(cache_path, "wb") as f:
        pickle.dump(cache, f)
    print(f"Saved inference results to {cache_path}")
    return cache
def plot_model_comparison(
    # out_preds,             # Dictionary: {model_name: {task_name: tensor}}
    # out_labels,            # Dictionary: {task_name: tensor} (ground truth + img)
    models_to_plot,        # List of model names (keys in out_preds) to plot
    input_gap_ratio=0.2,   # Width relative to a "normal" column (made slightly smaller)
    label_gap_ratio=0.2,   # Width relative to a "normal" column (made slightly smaller)
    input_col_ratio=1.0,   # Width ratio for the new input image column
    label_col_ratio=1.0,   # Width ratio for the label column
    model_col_ratio=1.0,   # Width ratio for each model prediction column
):
    """
    Plots a grid comparing model predictions against ground truth (LABEL)
    for various tasks. The input image is shown in the first column,
    centered vertically.
    """
    # imgs = {}
    # out_preds = {
    #     "fire": {}, 
    #     "worldfloods": {},
    #     "burned_area": {},
    #     "clouds": {}
    # }
    # out_labels = {
    #     "fire": {}, 
    #     "worldfloods": {},
    #     "burned_area": {},
    #     "clouds": {}
    # }   
    # for task, conf_dir in configs_dirs.items():
    #     first = True
    #     for model_final_name, model_name in model_configs.items():
    #         #try:
    #         configs_file = os.path.join(os.getcwd(), conf_dir, f"{model_name}.yml")
    #         model, dl_test = get_inference_model_and_dataset(model_final_name, task, str(configs_file))
    #         img = dl_test.dataset[1000]['img']
    #         out_label = dl_test.dataset[1000]['label']
    #         image, out_pred = return_inference_visualization(img, out_label, task, model)
    #         if first:
    #             imgs[task] = image
    #             out_labels[task] = out_label
    #             first = False
    #         out_preds[task][model_name] = out_pred
    #         #except Exception as e:
    #         #    print(f"Error loading model {model_name} from {configs_file}: {e}")
    #         #    continue
    figures = []
    cache_dicts = make_inference()
    samples = cache_dicts.keys()
    for sample in samples:
        imgs = cache_dicts[sample]["imgs"]
        print(imgs)
        out_preds = cache_dicts[sample]["out_preds"]
        out_labels = cache_dicts[sample]["out_labels"]
        # Define the order of tasks to display as rows
        task_order_display = task_order.copy()  
        task_order_display = [t for t in task_order_display if t in out_labels]
        n_rows = len(task_order_display)
        n_models = len(models_to_plot)
        if n_rows == 0:
            print("No tasks found in out_labels to display.")
            return

        # ---- build the list of width-ratios for GridSpec ---------------------------
        # 1 col Input + 1 small gap + 1 col Label + 1 small gap + N cols Models
        width_ratios = (
            [input_col_ratio] +      # Input Image Column
            [input_gap_ratio] +      # Gap Column 1
            [label_col_ratio] +      # Label Column
            [label_gap_ratio] +      # Gap Column 2
            [model_col_ratio] * n_models  # Model Columns
        )
        total_cols = len(width_ratios)  # Should now be 4 + n_models

        # ---- create figure & GridSpec ---------------------------------------------
        figsize_width  = max(3 * np.sum(width_ratios), 15)
        figsize_height = max(3 * n_rows, 8)
        fig = plt.figure(figsize=(figsize_width, figsize_height), constrained_layout=True)
        fig.set_constrained_layout_pads(
            w_pad=0.01,    # horizontal pad between axes, in fraction of axis size
            h_pad=0.01,    # vertical pad
            hspace=0,      # additional vertical space
            wspace=0       # additional horizontal space
        )

        
        gs  = fig.add_gridspec(n_rows, total_cols, width_ratios=width_ratios)


        # ---- Column indices -------------------------------------------------------
        col_input    = 0
        col_gap1     = 1
        col_label    = 2
        col_gap2     = 3
        col_model_0  = 4


        # ---- Prepare grid of axes (skip gap columns) ------------------------------
        axes_grid = [[None]*total_cols for _ in range(n_rows)]
        print(axes_grid)
        for r in range(n_rows):
            # Label column
            axes_grid[r][col_label] = fig.add_subplot(gs[r, col_label])
            axes_grid[r][col_label].axis("off")
            # Gaps are automatically empty
            axes_grid[r][col_gap1] = None
            axes_grid[r][col_gap2] = None
            # Model columns
            for j in range(n_models):
                c = (col_model_0-1 if j != 0 else col_input) + j
                if c >= total_cols:
                    break
                axes_grid[r][c] = fig.add_subplot(gs[r, c])
                axes_grid[r][c].axis("off")
                #axes_grid[r][c].set_title(task_labels.get("img", "Input Image"))
                if j == 0:
                    print(f"Shape of image for task {task_order_display[r]}: {imgs[task_order_display[r]].shape}")
                    plotters["img"](axes_grid[r][c], imgs[task_order_display[r]])

        # ---- Set column titles ----------------------------------------------------
        if n_rows > 0:
            axes_grid[0][0].set_title(task_labels.get("img", "Input Image"))
            axes_grid[0][col_label].set_title("Label")
            for j, m in enumerate(models_to_plot):
                if axes_grid[0][col_model_0 + j] is not None:
                    axes_grid[0][col_model_0 + j].set_title(model_labels.get(m, m))
        #print(f"Axes grid: {axes_grid}")
        # ---- Fill in each cell ----------------------------------------------------
        for r, task in enumerate(task_order_display):
            y_legend_base = -0.09 + (n_rows - r) * 0.2# Base vertical position (fraction of figure height from bottom)
            legend_height = 0.07 # Relative height for legends/colorbars
            # Ground truth
            ax_lbl = axes_grid[r][col_label]
            if task in out_labels:
                plotters[task](ax_lbl, out_labels[task])
            else:
                ax_lbl.text(0.5, 0.5, "Label N/A", ha="center", va="center", transform=ax_lbl.transAxes)

            # Predictions
            for j, m in enumerate(models_to_plot):
                ax_p = axes_grid[r][col_model_0 + j]
                if task in out_preds.keys() and m in out_preds[task]: #and task in out_preds[m]:
                    print(f"Prediction of task {task} by model {m}: {out_preds[task][m]}")
                    plotters[task](ax_p, out_preds[task][m])
                elif ax_p is not None:
                    ax_p.text(0.5, 0.5, "Pred. N/A", ha="center", va="center", transform=ax_p.transAxes)

            # Task annotation
            ax_lbl.annotate(
                task_labels.get(task, task),
                xy=(-0.27/label_col_ratio, 0.5),
                xycoords="axes fraction",
                va="center",
                ha="center",
                rotation="vertical",
            )
            x_lc = 0.15 # Start near left edge
            width_lc = 0.67 # Relative width for legend
            ax_lc = fig.add_axes([x_lc, y_legend_base, width_lc, legend_height])
            ax_lc.axis("off")
            handles = [patches.Patch(facecolor=colors[task][i], edgecolor="black") for i in codes]
            ax_lc.legend(
                handles,
                labels[task],
                ncol=min(6, len(labels[task])), # Adjust ncol based on number of labels
                loc="center",
                frameon=True,
                borderpad=0.5,      # Reduced padding
                handlelength=1.0,   # Reduced handle length
                columnspacing=1.0,  # Spacing between columns
                fontsize=legend_fontsize,
            )


        # ---------------- Legend & Colour-bars -------------------------------------
        # Adjust placement relative to the bottom of the figure
        # These might need fine-tuning based on the final figure aspect ratio and content


        # Calculate available width for legends/colorbars at the bottom
        # This depends on the figure width and the constrained_layout behavior
        # For simplicity, using relative positions and widths based on figure fraction


        # Colorbars
        # gap_cb_1 = -0.02 # Gap between LC legend and first colorbar, and between colorbars
        # gap_cb_2 = 0.03 # Gap between LC legend and first colorbar, and between colorbars
        # width_cb = (1- width_lc - gap_cb_1 - gap_cb_2 - x_lc) / 2 # Remaining width for colorbars

        # x_build = x_lc + width_lc + gap_cb_1
        # x_road = x_build + width_cb + gap_cb_2

        # # Check if colorbars might overflow and adjust widths if necessary
        # if x_road + width_cb > 0.98: # If right edge goes beyond 98% of figure width
        #    overflow = (x_road + width_cb) - 0.98
        #    # Reduce widths proportionally (simple approach)
        #    total_cb_width_area = width_lc + width_cb + width_cb + gap_cb_1 + gap_cb_2
        #    scale_factor = (total_cb_width_area - overflow) / total_cb_width_area
        #    width_lc *= scale_factor
        #    width_cb *= scale_factor
        #    gap_cb_1 *= scale_factor
        #    gap_cb_2 *= scale_factor
        #    # Recalculate positions
        #    x_build = x_lc + width_lc + gap_cb_1
        #    x_road = x_build + width_cb + gap_cb_2
        #    print("Warning: Adjusting legend/colorbar widths to fit figure.")

        # y_cb = y_legend_base + (legend_height/2 - 0.015) + 0.02

        # # Building density colorbar
        # cax_build = fig.add_axes([x_build, y_cb, width_cb, 0.03]) # Centered vertically with legend
        # sm_build = mpl.cm.ScalarMappable(cmap="YlOrRd", norm=mpl.colors.Normalize(0, 1))
        # sm_build.set_array([])
        # cb1 = fig.colorbar(sm_build, cax=cax_build, orientation="horizontal")
        # cb1.set_label("Building Density", labelpad=3, fontsize=legend_fontsize)
        # cb1.set_ticks([0, 0.5, 1])
        # cb1.ax.tick_params(labelsize=legend_fontsize)

        # # Road density colorbar
        # cax_road = fig.add_axes([x_road, y_cb, width_cb, 0.03]) # Centered vertically with legend
        # sm_road = mpl.cm.ScalarMappable(cmap="PuBu", norm=mpl.colors.Normalize(0, 1))
        # sm_road.set_array([])
        # cb2 = fig.colorbar(sm_road, cax=cax_road, orientation="horizontal")
        # cb2.set_label("Road Density", labelpad=3, fontsize=legend_fontsize)
        # cb2.set_ticks([0, 0.5, 1])
        # cb2.ax.tick_params(labelsize=legend_fontsize)

        plt.show()
        figures.append(fig)
        #return fig
    return figures


models_to_plot = sorted(model_labels.keys())


figures = plot_model_comparison(
    models_to_plot=models_to_plot
)

for i, fig in enumerate(figures):
    fig.savefig(f"model_comparison_{i}.pdf", format="pdf", bbox_inches="tight", dpi=500)

In [ ]:
import pickle
import copy 

def make_inference(cache_path="inference_cache.pkl"):
    # Check if cached results exist
    
    if os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            cache = pickle.load(f)
        print(f"Loaded inference results from {cache_path}")
        #return cache
    else:
        cache = {}
    cache_entry =             {
        "imgs": {}, 
        "out_preds": {
            "fire": {}, 
            "worldfloods": {},
            "burned_area": {},
            "clouds": {}
        }, 
        "out_labels":  {
            "fire": {}, 
            "worldfloods": {},
            "burned_area": {},
            "clouds": {}
        }
    }


    sample_nums = [50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000]

    for task, conf_dir in configs_dirs.items():
        first = True
        for model_final_name, model_name in model_configs.items():
            model, dl_test = None, None
            for sample_num in sample_nums: 
                if sample_num not in cache.keys():
                    cache[sample_num] = copy.deepcopy(cache_entry)
                if model_name not in cache[sample_num]['out_preds'][task].keys() or cache[sample_num]['imgs'][task] is None:
                    if model is None and dl_test is None:
                        configs_file = os.path.join(os.getcwd(), conf_dir, f"{model_name}.yml")
                        model, dl_test = get_inference_model_and_dataset(model_final_name, task, str(configs_file))
                    try:
                        img = dl_test.dataset[sample_num]['img']
                        out_label = dl_test.dataset[sample_num]['label']
                        image, out_pred = return_inference_visualization(img, out_label, task, model)
                        #if first:
                        # print(f"Shape of image for task {task}: {image.shape}")
                        # cache[sample_num]['imgs'][task] = image
                        # cache[sample_num]['out_labels'][task] = out_label
                        #first = False
                        cache[sample_num]['out_preds'][task][model_name] = out_pred
                    except Exception as e:
                        print(f"Error loading model {model_name} from {configs_file}: {e}")  
                              
    # imgs = {sample_id: cache_e['imgs'] for sample_id, cache_e in cache.items()}
    # out_preds = {sample_id: cache_e['out_preds'] for sample_id, cache_e in cache.items()}
    # out_labels = {sample_id: cache_e['out_labels'] for sample_id, cache_e in cache.items()}
    #cache = {"imgs": imgs, "out_preds": out_preds, "out_labels": out_labels}
    with open(cache_path, "wb") as f:
        pickle.dump(cache, f)
    print(f"Saved inference results to {cache_path}")
    return cache
def plot_model_comparison(
    # out_preds,             # Dictionary: {model_name: {task_name: tensor}}
    # out_labels,            # Dictionary: {task_name: tensor} (ground truth + img)
    models_to_plot,        # List of model names (keys in out_preds) to plot
    input_gap_ratio=0.2,   # Width relative to a "normal" column (made slightly smaller)
    label_gap_ratio=0.2,   # Width relative to a "normal" column (made slightly smaller)
    input_col_ratio=1.0,   # Width ratio for the new input image column
    label_col_ratio=1.0,   # Width ratio for the label column
    model_col_ratio=1.0,   # Width ratio for each model prediction column
):
    """
    Plots a grid comparing model predictions against ground truth (LABEL)
    for various tasks. The input image is shown in the first column,
    centered vertically.
    """
    # imgs = {}
    # out_preds = {
    #     "fire": {}, 
    #     "worldfloods": {},
    #     "burned_area": {},
    #     "clouds": {}
    # }
    # out_labels = {
    #     "fire": {}, 
    #     "worldfloods": {},
    #     "burned_area": {},
    #     "clouds": {}
    # }   
    # for task, conf_dir in configs_dirs.items():
    #     first = True
    #     for model_final_name, model_name in model_configs.items():
    #         #try:
    #         configs_file = os.path.join(os.getcwd(), conf_dir, f"{model_name}.yml")
    #         model, dl_test = get_inference_model_and_dataset(model_final_name, task, str(configs_file))
    #         img = dl_test.dataset[1000]['img']
    #         out_label = dl_test.dataset[1000]['label']
    #         image, out_pred = return_inference_visualization(img, out_label, task, model)
    #         if first:
    #             imgs[task] = image
    #             out_labels[task] = out_label
    #             first = False
    #         out_preds[task][model_name] = out_pred
    #         #except Exception as e:
    #         #    print(f"Error loading model {model_name} from {configs_file}: {e}")
    #         #    continue
    figures = []
    cache_dicts = make_inference()
    samples = cache_dicts.keys()
    for sample in samples:
        imgs = cache_dicts[sample]["imgs"]
        print(imgs)
        out_preds = cache_dicts[sample]["out_preds"]
        out_labels = cache_dicts[sample]["out_labels"]
        # Define the order of tasks to display as rows
        task_order_display = task_order.copy()  
        task_order_display = [t for t in task_order_display if t in out_labels]
        n_rows = len(task_order_display)
        n_models = len(models_to_plot)
        if n_rows == 0:
            print("No tasks found in out_labels to display.")
            return

        # ---- build the list of width-ratios for GridSpec ---------------------------
        # 1 col Input + 1 small gap + 1 col Label + 1 small gap + N cols Models
        width_ratios = (
            [input_col_ratio] +      # Input Image Column
            [input_gap_ratio] +      # Gap Column 1
            [label_col_ratio] +      # Label Column
            [label_gap_ratio] +      # Gap Column 2
            [model_col_ratio] * n_models  # Model Columns
        )
        total_cols = len(width_ratios)  # Should now be 4 + n_models

        # ---- create figure & GridSpec ---------------------------------------------
        figsize_width  = max(3 * np.sum(width_ratios), 15)
        figsize_height = max(3 * n_rows, 8)
        fig = plt.figure(figsize=(figsize_width, figsize_height), constrained_layout=True)
        fig.set_constrained_layout_pads(
            w_pad=0.01,    # horizontal pad between axes, in fraction of axis size
            h_pad=0.01,    # vertical pad
            hspace=0,      # additional vertical space
            wspace=0       # additional horizontal space
        )

        
        gs  = fig.add_gridspec(n_rows, total_cols, width_ratios=width_ratios)


        # ---- Column indices -------------------------------------------------------
        col_input    = 0
        col_gap1     = 1
        col_label    = 2
        col_gap2     = 3
        col_model_0  = 4


        # ---- Prepare grid of axes (skip gap columns) ------------------------------
        axes_grid = [[None]*total_cols for _ in range(n_rows)]
        print(axes_grid)
        for r in range(n_rows):
            # Label column
            axes_grid[r][col_label] = fig.add_subplot(gs[r, col_label])
            axes_grid[r][col_label].axis("off")
            # Gaps are automatically empty
            axes_grid[r][col_gap1] = None
            axes_grid[r][col_gap2] = None
            # Model columns
            for j in range(n_models):
                c = (col_model_0-1 if j != 0 else col_input) + j
                if c >= total_cols:
                    break
                axes_grid[r][c] = fig.add_subplot(gs[r, c])
                axes_grid[r][c].axis("off")
                #axes_grid[r][c].set_title(task_labels.get("img", "Input Image"))
                if j == 0:
                    print(f"Shape of image for task {task_order_display[r]}: {imgs[task_order_display[r]].shape}")
                    plotters["img"](axes_grid[r][c], imgs[task_order_display[r]])

        # ---- Set column titles ----------------------------------------------------
        if n_rows > 0:
            axes_grid[0][0].set_title(task_labels.get("img", "Input Image"))
            axes_grid[0][col_label].set_title("Label")
            for j, m in enumerate(models_to_plot):
                if axes_grid[0][col_model_0 + j] is not None:
                    axes_grid[0][col_model_0 + j].set_title(model_labels.get(m, m))
        #print(f"Axes grid: {axes_grid}")
        # ---- Fill in each cell ----------------------------------------------------
        for r, task in enumerate(task_order_display):
            y_legend_base = -0.09 + (n_rows - r) * 0.2# Base vertical position (fraction of figure height from bottom)
            legend_height = 0.07 # Relative height for legends/colorbars
            # Ground truth
            ax_lbl = axes_grid[r][col_label]
            if task in out_labels:
                plotters[task](ax_lbl, out_labels[task])
            else:
                ax_lbl.text(0.5, 0.5, "Label N/A", ha="center", va="center", transform=ax_lbl.transAxes)

            # Predictions
            for j, m in enumerate(models_to_plot):
                ax_p = axes_grid[r][col_model_0 + j]
                if task in out_preds.keys() and m in out_preds[task]: #and task in out_preds[m]:
                    print(f"Prediction of task {task} by model {m}: {out_preds[task][m]}")
                    plotters[task](ax_p, out_preds[task][m])
                elif ax_p is not None:
                    ax_p.text(0.5, 0.5, "Pred. N/A", ha="center", va="center", transform=ax_p.transAxes)

            # Task annotation
            ax_lbl.annotate(
                task_labels.get(task, task),
                xy=(-0.27/label_col_ratio, 0.5),
                xycoords="axes fraction",
                va="center",
                ha="center",
                rotation="vertical",
            )
            x_lc = 0.15 # Start near left edge
            width_lc = 0.67 # Relative width for legend
            ax_lc = fig.add_axes([x_lc, y_legend_base, width_lc, legend_height])
            ax_lc.axis("off")
            handles = [patches.Patch(facecolor=colors[task][i], edgecolor="black") for i in codes]
            ax_lc.legend(
                handles,
                labels[task],
                ncol=min(6, len(labels[task])), # Adjust ncol based on number of labels
                loc="center",
                frameon=True,
                borderpad=0.5,      # Reduced padding
                handlelength=1.0,   # Reduced handle length
                columnspacing=1.0,  # Spacing between columns
                fontsize=legend_fontsize,
            )


        # ---------------- Legend & Colour-bars -------------------------------------
        # Adjust placement relative to the bottom of the figure
        # These might need fine-tuning based on the final figure aspect ratio and content


        # Calculate available width for legends/colorbars at the bottom
        # This depends on the figure width and the constrained_layout behavior
        # For simplicity, using relative positions and widths based on figure fraction


        # Colorbars
        # gap_cb_1 = -0.02 # Gap between LC legend and first colorbar, and between colorbars
        # gap_cb_2 = 0.03 # Gap between LC legend and first colorbar, and between colorbars
        # width_cb = (1- width_lc - gap_cb_1 - gap_cb_2 - x_lc) / 2 # Remaining width for colorbars

        # x_build = x_lc + width_lc + gap_cb_1
        # x_road = x_build + width_cb + gap_cb_2

        # # Check if colorbars might overflow and adjust widths if necessary
        # if x_road + width_cb > 0.98: # If right edge goes beyond 98% of figure width
        #    overflow = (x_road + width_cb) - 0.98
        #    # Reduce widths proportionally (simple approach)
        #    total_cb_width_area = width_lc + width_cb + width_cb + gap_cb_1 + gap_cb_2
        #    scale_factor = (total_cb_width_area - overflow) / total_cb_width_area
        #    width_lc *= scale_factor
        #    width_cb *= scale_factor
        #    gap_cb_1 *= scale_factor
        #    gap_cb_2 *= scale_factor
        #    # Recalculate positions
        #    x_build = x_lc + width_lc + gap_cb_1
        #    x_road = x_build + width_cb + gap_cb_2
        #    print("Warning: Adjusting legend/colorbar widths to fit figure.")

        # y_cb = y_legend_base + (legend_height/2 - 0.015) + 0.02

        # # Building density colorbar
        # cax_build = fig.add_axes([x_build, y_cb, width_cb, 0.03]) # Centered vertically with legend
        # sm_build = mpl.cm.ScalarMappable(cmap="YlOrRd", norm=mpl.colors.Normalize(0, 1))
        # sm_build.set_array([])
        # cb1 = fig.colorbar(sm_build, cax=cax_build, orientation="horizontal")
        # cb1.set_label("Building Density", labelpad=3, fontsize=legend_fontsize)
        # cb1.set_ticks([0, 0.5, 1])
        # cb1.ax.tick_params(labelsize=legend_fontsize)

        # # Road density colorbar
        # cax_road = fig.add_axes([x_road, y_cb, width_cb, 0.03]) # Centered vertically with legend
        # sm_road = mpl.cm.ScalarMappable(cmap="PuBu", norm=mpl.colors.Normalize(0, 1))
        # sm_road.set_array([])
        # cb2 = fig.colorbar(sm_road, cax=cax_road, orientation="horizontal")
        # cb2.set_label("Road Density", labelpad=3, fontsize=legend_fontsize)
        # cb2.set_ticks([0, 0.5, 1])
        # cb2.ax.tick_params(labelsize=legend_fontsize)

        plt.show()
        figures.append(fig)
        #return fig
    return figures


models_to_plot = sorted(model_labels.keys())


figures = plot_model_comparison(
    models_to_plot=models_to_plot
)

for i, fig in enumerate(figures):
    fig.savefig(f"model_comparison_{i}.pdf", format="pdf", bbox_inches="tight", dpi=500)
